# 01 · Data Types: Inspect & Fix

**Goal:** load the Telco Customer Churn data, find out what data type every column *currently* has, spot the ones that are wrong, and fix them.

1. **Find** the data types (and check whether they match what the values really are)
2. **Fix** them
3. **Verify** and save a clean copy for the next notebooks

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

RAW_PATH = "WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(RAW_PATH)
print(f"Rows: {df.shape[0]:,}   Columns: {df.shape[1]}")
df.head()

Rows: 7,043   Columns: 21


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Step 1 · Find the data types

### 1a. What pandas thinks each column is

In [2]:
dtypes_before = df.dtypes.copy()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

### 1b. What the values actually look like

For each column: current dtype, number of unique values, and a few example values. This tells us whether a column is really numeric, binary (Yes/No), categorical, or an ID.

In [3]:
profile = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "n_unique": df.nunique(),
    "n_missing": df.isna().sum(),
    "examples": [df[c].dropna().unique()[:5].tolist() for c in df.columns],
})
profile

,dtype,n_unique,n_missing,examples
customerID,str,7043,0,"[7590-VHVEG, 5575-GNVDE, 3668-QPYBK, 7795-CFOC..."
gender,str,2,0,"[Female, Male]"
SeniorCitizen,int64,2,0,"[0, 1]"
Partner,str,2,0,"[Yes, No]"
Dependents,str,2,0,"[No, Yes]"
tenure,int64,73,0,"[1, 34, 2, 45, 8]"
PhoneService,str,2,0,"[No, Yes]"
MultipleLines,str,3,0,"[No phone service, No, Yes]"
InternetService,str,3,0,"[DSL, Fiber optic, No]"
OnlineSecurity,str,3,0,"[No, Yes, No internet service]"


### 1c. Detect problems automatically

Checks:
- **Hidden blanks:** text cells that are empty or only spaces (pandas does *not* count these as missing)
- **Numbers stored as text:** text columns (`object` / `str`) where almost every value converts to a number
- **Binary columns:** only 2 unique values
- **Numeric codes that are really categories:** integer columns with only 0/1

In [4]:
def is_text(s):
    # pandas 2 stores text as 'object', pandas 3 as 'str'; handle both
    return pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s)

issues = []
for col in df.columns:
    s = df[col]
    if col == "customerID":
        issues.append((col, "unique per row -> identifier, not a feature"))
    elif is_text(s):
        blanks = (s.str.strip() == "").sum()
        numeric_share = pd.to_numeric(s, errors="coerce").notna().mean()
        if blanks:
            issues.append((col, f"{blanks} hidden blank value(s)"))
        if numeric_share > 0.95:
            issues.append((col, f"stored as text but {numeric_share:.1%} of values are numbers -> should be numeric"))
        elif s.nunique() == 2:
            issues.append((col, f"binary text {sorted(s.unique())} -> category"))
        elif s.nunique() <= 10:
            issues.append((col, f"{s.nunique()} text levels -> category"))
    elif pd.api.types.is_integer_dtype(s) and set(s.unique()) <= {0, 1}:
        issues.append((col, "integer 0/1 flag -> really a Yes/No category"))

pd.DataFrame(issues, columns=["column", "finding"])

,column,finding
0,customerID,"unique per row -> identifier, not a feature"
1,gender,"binary text ['Female', 'Male'] -> category"
2,SeniorCitizen,integer 0/1 flag -> really a Yes/No category
3,Partner,"binary text ['No', 'Yes'] -> category"
4,Dependents,"binary text ['No', 'Yes'] -> category"
5,PhoneService,"binary text ['No', 'Yes'] -> category"
6,MultipleLines,3 text levels -> category
7,InternetService,3 text levels -> category
8,OnlineSecurity,3 text levels -> category
9,OnlineBackup,3 text levels -> category


### 1d. Look closer at `TotalCharges`

It should be a money amount, but pandas read it as text. Which rows break the conversion?

In [5]:
bad_total = pd.to_numeric(df["TotalCharges"], errors="coerce").isna()
print(f"Rows where TotalCharges is not a number: {bad_total.sum()}")
df.loc[bad_total, ["customerID", "tenure", "MonthlyCharges", "TotalCharges"]]

Rows where TotalCharges is not a number: 11


,customerID,tenure,MonthlyCharges,TotalCharges
488,4472-LVYGI,0,52.55,
753,3115-CZMZD,0,20.25,
936,5709-LVOEQ,0,80.85,
1082,4367-NUYAO,0,25.75,
1340,1371-DWPAZ,0,56.05,
3331,7644-OMVMY,0,19.85,
3826,3213-VVOLG,0,25.35,
4380,2520-SGTTA,0,20.00,
5218,2923-ARZLG,0,19.70,
6670,4075-WKNIU,0,73.35,


**Finding:** every broken row has `tenure = 0`. These are brand-new customers who haven't been billed yet, so their total charges are really **0**, not unknown.

## Step 2 · Fix the data types

| Column(s) | Problem | Fix |
|---|---|---|
| `TotalCharges` | text with blank values | convert to float, blanks -> 0 (tenure is 0) |
| `SeniorCitizen` | 0/1 integer, but every other Yes/No column is text | map to `No`/`Yes`, then category |
| `Churn` | Yes/No text | 0/1 integer (the **target** the model predicts) |
| other text columns | plain text (`object` / `str`) | `category` (less memory, clear that they're categories) |
| `customerID` | plain text | `string`: it's an identifier, not a feature |

In [6]:
clean = df.copy()

# 1. TotalCharges: text -> number; blanks belong to tenure-0 customers, so they owe 0
clean["TotalCharges"] = pd.to_numeric(clean["TotalCharges"].str.strip(), errors="coerce")
assert (clean.loc[clean["TotalCharges"].isna(), "tenure"] == 0).all(), "unexpected missing TotalCharges"
clean["TotalCharges"] = clean["TotalCharges"].fillna(0.0)

# 2. SeniorCitizen: 0/1 -> No/Yes so it matches the other binary columns
clean["SeniorCitizen"] = clean["SeniorCitizen"].map({0: "No", 1: "Yes"})

# 3. Churn: target variable -> 0/1
clean["Churn"] = clean["Churn"].map({"No": 0, "Yes": 1}).astype("int8")

# 4. customerID: identifier
clean["customerID"] = clean["customerID"].astype("string")

# 5. All remaining text columns -> category
cat_cols = [c for c in clean.columns if c != "customerID" and is_text(clean[c])]
clean[cat_cols] = clean[cat_cols].astype("category")

# 6. tenure: whole months, small range -> compact integer
clean["tenure"] = clean["tenure"].astype("int16")

print("Converted to category:", cat_cols)

Converted to category: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


## Step 3 · Verify

### 3a. Before vs after

In [7]:
comparison = pd.DataFrame({"before": dtypes_before.astype(str), "after": clean.dtypes.astype(str)})
comparison["changed"] = np.where(comparison["before"] != comparison["after"], "yes", "")
comparison

,before,after,changed
customerID,str,string,yes
gender,str,category,yes
SeniorCitizen,int64,category,yes
Partner,str,category,yes
Dependents,str,category,yes
tenure,int64,int16,yes
PhoneService,str,category,yes
MultipleLines,str,category,yes
InternetService,str,category,yes
OnlineSecurity,str,category,yes


### 3b. Sanity checks

In [8]:
print("Missing values left:", int(clean.isna().sum().sum()))
print("Duplicate customer IDs:", int(clean["customerID"].duplicated().sum()))
print(f"Churn rate: {clean['Churn'].mean():.1%}")

# TotalCharges should be roughly tenure x MonthlyCharges
approx = clean["tenure"] * clean["MonthlyCharges"]
print(f"Correlation TotalCharges vs tenure*MonthlyCharges: {clean['TotalCharges'].corr(approx):.3f}")

mem_before = df.memory_usage(deep=True).sum() / 1024**2
mem_after = clean.memory_usage(deep=True).sum() / 1024**2
print(f"Memory: {mem_before:.2f} MB -> {mem_after:.2f} MB")

clean.describe()

Missing values left: 0
Duplicate customer IDs: 0
Churn rate: 26.5%
Correlation TotalCharges vs tenure*MonthlyCharges: 1.000
Memory: 1.86 MB -> 0.36 MB


,tenure,MonthlyCharges,TotalCharges,Churn
count,7043.000000,7043.000000,7043.000000,7043.000000
mean,32.371149,64.761692,2279.734304,0.265370
std,24.559481,30.090047,2266.794470,0.441561
min,0.000000,18.250000,0.000000,0.000000
25%,9.000000,35.500000,398.550000,0.000000
50%,29.000000,70.350000,1394.550000,0.000000
75%,55.000000,89.850000,3786.600000,1.000000
max,72.000000,118.750000,8684.800000,1.000000


### 3c. Category levels (check for typos / odd values)

In [9]:
for col in clean.select_dtypes(include="category").columns:
    print(f"{col:18s} {list(clean[col].cat.categories)}")

gender             ['Female', 'Male']
SeniorCitizen      ['No', 'Yes']
Partner            ['No', 'Yes']
Dependents         ['No', 'Yes']
PhoneService       ['No', 'Yes']
MultipleLines      ['No', 'No phone service', 'Yes']
InternetService    ['DSL', 'Fiber optic', 'No']
OnlineSecurity     ['No', 'No internet service', 'Yes']
OnlineBackup       ['No', 'No internet service', 'Yes']
DeviceProtection   ['No', 'No internet service', 'Yes']
TechSupport        ['No', 'No internet service', 'Yes']
StreamingTV        ['No', 'No internet service', 'Yes']
StreamingMovies    ['No', 'No internet service', 'Yes']
Contract           ['Month-to-month', 'One year', 'Two year']
PaperlessBilling   ['No', 'Yes']
PaymentMethod      ['Bank transfer (automatic)', 'Credit card (automatic)', 'Electronic check', 'Mailed check']


## Save the clean data

Saved as **pickle** so the next notebooks load it with the fixed types intact (a CSV would turn categories back into plain text). A CSV copy is also saved for viewing in Excel.

In [10]:
clean.to_pickle("telco_clean.pkl")
clean.to_csv("telco_clean.csv", index=False)
print("Saved telco_clean.pkl and telco_clean.csv")

# Next notebooks: df = pd.read_pickle("telco_clean.pkl")

Saved telco_clean.pkl and telco_clean.csv
